# 26 â€” Diverse Bagging Ensemble for Deployable Firewall

**Objective**: Implement "Diverse Bagging" strategy for production-ready VPN firewall with Zero-FP guarantee.

## Strategy:
1. **Robust Features**
2. **Sub-Capture Splitting**
3. **DIVERSE BAGGING**
4. **Forensic Validation**

In [1]:
%matplotlib inline

import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

sys.path.append(str(Path.cwd().parent))

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline
from src.models.train_balanced_bagging_ensemble import run_balanced_bagging

paths = load_paths()
logger = setup_logger(level="INFO")

SEED = 42

COMPACT_FEATURES = [
    "sz_coef_variation",
    "sz_p25_median_ratio",
    "sz_p75_median_ratio",
    "sz_iqr_norm_median",
    "dispersion_symmetry",
    "direction_balance_bytes",
    "direction_balance_packets"
]


# NOTE: IAT features moved to experimental (dataset leakage detected - AUC = 0.93)
IAT_EXPERIMENTAL_FEATURES = [
    "iat_all_mean",
    "iat_all_std",
    "iat_all_p25",
    "iat_all_median",
    "iat_all_p75",
    "iat_mean_max",
    "iat_mean_min",
    "iat_std_max",
    "iat_std_min"
]

print(f"Using Compact feature set: 12 features (excluding IAT and duplicates)")
print(f"IAT features available for experimental use: 9 features")


Using Compact feature set: 12 features (excluding IAT and duplicates)
IAT features available for experimental use: 9 features


## 1. Load Multi-Domain Data

In [2]:
df_all = load_and_prepare_data()

print(f"Multi-Domain Pool: {df_all.shape}")
print("\nDatasets by split:")
print(pd.crosstab(df_all["split"], df_all["dataset"]))

print("\nLabel distribution:")
print(pd.crosstab(df_all["dataset"], df_all["label"], margins=True))

df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()
df_test = df_all[df_all["split"] == "test"].copy()

pipe = FeaturePipeline().fit(df_train)
all_features = pipe.model_feature_names()

missing_compact = [f for f in COMPACT_FEATURES if f not in all_features]
if missing_compact:
    raise ValueError(f"Missing Robust features: {missing_compact}")

print(f"\nFeature pipeline fitted with {len(all_features)} features")
print(f"Compact feature set confirmed present: {all(f in all_features for f in COMPACT_FEATURES)}")

2026-03-30 12:37:03 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:37:03 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-30 12:37:04 | INFO | ai-vpn-firewall | Removed 5750 duplicate flows (7.92%)
2026-03-30 12:37:04 | INFO | ai-vpn-firewall | Metadata columns present for analysis only: ['source_capture_id', 'source_file']
2026-03-30 12:37

## 2. Train Balanced Bagging Ensemble

In [3]:
df_train_transformed = pipe.transform(df_train)
df_val_transformed = pipe.transform(df_val)
df_test_transformed = pipe.transform(df_test)

df_all_transformed = pd.concat(
    [df_train_transformed, df_val_transformed, df_test_transformed],
    ignore_index=True
)

print(f"Transformed data shape: {df_all_transformed.shape}")
print("Training ensemble on ROBUST features only...")

output_dir = paths.artifacts_ensemble / "diverse_bagging_robust"
output_dir.mkdir(parents=True, exist_ok=True)

results = run_balanced_bagging(
    df=df_all_transformed,
    label_col="label",
    group_col="capture_id",
    dataset_col="dataset",
    split_col="split",
    diverse_ratios=[1.0, 5.0, 10.0],
    target_fprs="0.001,0.005,0.01",
    seed=SEED,
    output_dir=str(output_dir),
    model_types=["xgb", "lgbm", "cat"],
    feature_cols=COMPACT_FEATURES,
    weight_xgb=1.0,
    weight_lgbm=1.0,
    weight_cat=1.0,
)

print("\n" + "="*80)
print("BALANCED ENSEMBLE TRAINING COMPLETE")
print("="*80)

Transformed data shape: (66862, 15)
Training ensemble on ROBUST features only...

BALANCED ENSEMBLE TRAINING COMPLETE


## 3. Forensic Validation: Leave-One-Dataset-Out (LOOD)

In [4]:
def run_lood_test(df_all_in, pipe, feature_cols, seed=42):

    datasets = sorted(df_all_in["dataset"].unique())
    results = []

    for held_out in datasets:

        print(f"\n{'='*60}")
        print(f"LOOD: Holding out {held_out.upper()}")
        print(f"{'='*60}")

        train_mask = (
            (df_all_in["split"] == "train")
            & (df_all_in["dataset"] != held_out)
        )

        val_mask = (
            (df_all_in["split"] == "val")
            & (df_all_in["dataset"] != held_out)
        )

        test_mask = (
            (df_all_in["split"] == "test")
            & (df_all_in["dataset"] == held_out)
        )

        df_train_lo = df_all_in[train_mask].copy()
        df_val_lo = df_all_in[val_mask].copy()
        df_test_lo = df_all_in[test_mask].copy()

        if len(df_train_lo) == 0 or len(df_test_lo) == 0:
            print(f"Skipping {held_out}: insufficient data")
            continue

        pipe_lo = FeaturePipeline().fit(df_train_lo)

        df_train_lo_t = pipe_lo.transform(df_train_lo)
        df_val_lo_t = pipe_lo.transform(df_val_lo)
        df_test_lo_t = pipe_lo.transform(df_test_lo)

        df_lo_combined = pd.concat(
            [df_train_lo_t, df_val_lo_t, df_test_lo_t],
            ignore_index=True
        )

        temp_dir = paths.artifacts_ensemble / f"lood_{held_out}"
        temp_dir.mkdir(parents=True, exist_ok=True)

        lood_results = run_balanced_bagging(
            df=df_lo_combined,
            label_col="label",
            group_col="capture_id",
            dataset_col="dataset",
            split_col="split",
            bags_per_family=3,
            majority_ratio=1.0,
            target_fprs="0.01",
            seed=seed,
            output_dir=str(temp_dir),
            model_types=["xgb", "lgbm", "cat"],
            feature_cols=feature_cols,
            weight_xgb=1.0,
            weight_lgbm=1.0,
            weight_cat=1.0,
        )

        test_metrics = lood_results["isotonic"]["test_overall"]

        results.append({
            "train_on": "+".join(sorted(
                [d for d in datasets if d != held_out]
            )),
            "test_on": held_out,
            "auc": test_metrics["auc"],
            "pr_auc": test_metrics["pr_auc"],
            "threshold": test_metrics["fpr_0.01"]["threshold"],
            "recall": test_metrics["fpr_0.01"]["recall"],
            "precision": test_metrics["fpr_0.01"]["precision"],
            "fpr": test_metrics["fpr_0.01"]["fpr"],
        })

    return pd.DataFrame(results)

In [5]:
print("\n" + "="*80)
print("FORENSIC VALIDATION: LEAVE-ONE-DATASET-OUT")
print("="*80)

lood_df = run_lood_test(df_all, pipe, COMPACT_FEATURES, seed=SEED)

print("\n" + "="*80)
print("LOOD RESULTS")
print("="*80)

print(lood_df.to_string(index=False, float_format="%.4f"))

mean_auc = lood_df["auc"].mean()
print(f"\n Mean LOOD AUC: {mean_auc:.4f}")

if mean_auc > 0.80:
    print("SUCCESS: Era-independent model achieved robust generalization (>0.80)")
else:
    print("WARNING: LOOD AUC still below 0.80 threshold")


FORENSIC VALIDATION: LEAVE-ONE-DATASET-OUT

LOOD: Holding out ISCX

LOOD: Holding out USBVPN

LOOD: Holding out VNAT

LOOD RESULTS
   train_on test_on    auc  pr_auc  threshold  recall  precision    fpr
usbvpn+vnat    iscx 0.5317  0.1908     0.0012  0.9870     0.1910 0.9226
  iscx+vnat  usbvpn 0.5519  0.0738     0.2649  0.3877     0.0861 0.2724
iscx+usbvpn    vnat 0.3779  0.0069     0.5207  0.0000     0.0000 0.0619

 Mean LOOD AUC: 0.4872


## 4. Detailed Test Results by Dataset

In [6]:
print("\n" + "="*80)
print("DETAILED TEST PERFORMANCE BY DATASET (Isotonic Calibration)")
print("="*80)

pred_df = pd.read_csv(output_dir / "predictions.csv")
test_pred = pred_df[pred_df["split"] == "test"].copy()

summary_rows = []

for ds in sorted(test_pred["dataset"].unique()):

    sub = test_pred[test_pred["dataset"] == ds]

    y_true = sub["label"].values
    y_prob = sub["prob_iso"].values

    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    with open(output_dir / "metrics.json", "r") as f:
        metrics = json.load(f)

    thr = metrics["isotonic"][f"test_{ds}"]["fpr_0.01"]["threshold"]

    y_pred = (y_prob >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    recall = tp / (tp + fn) if (tp + fn) else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0

    summary_rows.append({
        "dataset": ds,
        "auc": auc,
        "pr_auc": pr_auc,
        "threshold": thr,
        "recall": recall,
        "precision": precision,
        "fpr": fpr,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "n_test": len(y_true),
    })

summary_df = pd.DataFrame(summary_rows)

print(summary_df.to_string(index=False, float_format="%.4f"))


DETAILED TEST PERFORMANCE BY DATASET (Isotonic Calibration)
dataset    auc  pr_auc  threshold  recall  precision    fpr   tn  fp  fn  tp  n_test
   iscx 0.8458  0.4146     0.0578  0.3983     0.4600 0.1032  939 108 139  92    1278
 usbvpn 0.9937  0.9645     0.0578  0.9314     0.9657 0.0022 6373  14  29 394    6810
   vnat 0.9909  0.3222     0.0578  0.0000     0.0000 0.0025  403   1   3   0     407


In [7]:
print("\n" + "="*80)
print("ROBUST BALANCED ENSEMBLE: FINAL SUMMARY")
print("="*80)

print("\n1. STRATEGY IMPLEMENTATION:")
print(f"   âœ“ Robust Features: {COMPACT_FEATURES}")
print("   âœ“ USBVPN Sub-capture size: 100")
print("   âœ“ DIVERSE Bagging: 3 families Ã— 3 diverse bags = 9 models")
print("   âœ“ Bag 1 (Sensitive): 1:1 ratio")
print("   âœ“ Bag 2 (Balanced): 1:5 ratio")
print("   âœ“ Bag 3 (Conservative): 1:10 ratio")

print("\n2. FORENSIC VALIDATION (LOOD):")
print(lood_df[["train_on", "test_on", "auc"]].to_string(index=False))

print(f"\n   Mean LOOD ROC: {lood_df['auc'].mean():.4f}")
print(f"   Min LOOD AUC: {lood_df['auc'].min():.4f}")
print(f"   Max LOOD AUC: {lood_df['auc'].max():.4f}")

print("\n3. PER-DATASET TEST PERFORMANCE:")
print(summary_df[["dataset", "auc", "pr_auc", "recall", "precision", "fpr"]]
      .to_string(index=False))

print("\n4. LEAKAGE VERIFICATION:")

train_caps = set(df_train["capture_id"].unique())
test_caps = set(df_test["capture_id"].unique())

overlap = train_caps & test_caps

print(f"Train/Test capture_id overlap: {len(overlap)}")

if len(overlap) == 0:
    print("No leakage detected")
else:
    print("WARNING: leakage detected")

print("\n5. DELIVERABLES:")
print(f"Models saved to: {output_dir}")
print(f"Predictions: {output_dir / 'predictions.csv'}")
print(f"Metrics: {output_dir / 'metrics.json'}")

print("\n" + "="*80)

if lood_df["auc"].mean() > 0.80:
    print("MISSION ACCOMPLISHED")
else:
    print("Results logged. Further optimization needed.")

print("="*80)


ROBUST BALANCED ENSEMBLE: FINAL SUMMARY

1. STRATEGY IMPLEMENTATION:
   âœ“ Robust Features: ['sz_coef_variation', 'sz_p25_median_ratio', 'sz_p75_median_ratio', 'sz_iqr_norm_median', 'dispersion_symmetry', 'direction_balance_bytes', 'direction_balance_packets']
   âœ“ USBVPN Sub-capture size: 100
   âœ“ DIVERSE Bagging: 3 families Ã— 3 diverse bags = 9 models
   âœ“ Bag 1 (Sensitive): 1:1 ratio
   âœ“ Bag 2 (Balanced): 1:5 ratio
   âœ“ Bag 3 (Conservative): 1:10 ratio

2. FORENSIC VALIDATION (LOOD):
   train_on test_on      auc
usbvpn+vnat    iscx 0.531717
  iscx+vnat  usbvpn 0.551873
iscx+usbvpn    vnat 0.377888

   Mean LOOD ROC: 0.4872
   Min LOOD AUC: 0.3779
   Max LOOD AUC: 0.5519

3. PER-DATASET TEST PERFORMANCE:
dataset      auc   pr_auc   recall  precision      fpr
   iscx 0.845822 0.414559 0.398268   0.460000 0.103152
 usbvpn 0.993729 0.964508 0.931442   0.965686 0.002192
   vnat 0.990924 0.322222 0.000000   0.000000 0.002475

4. LEAKAGE VERIFICATION:
Train/Test capture_id ov